<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/8_16_26_Indigo_Random_Forest_YFinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:01<00:00, 69.8MB/s] 


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [5]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [7]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [8]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_1248/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*****                 10%                       ]  49 of 503 completedERROR:yfinance:Failed to get ticker 'KLAC' reason: Failed to perform, curl: (28) Connection timed out after 10002 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[********              16%                       ]  82 of 503 completedERROR:yfinance:Failed to get ticker 'PSKY' reason: Failed to perform, curl: (28) Connection timed out after 10003 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[**************        29%                       ]  144 of 503 completedERROR:yfinance:Failed to get ticker 'NXPI' reason: Failed to perform, curl: (28) Connection timed out after 10002 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[**************       

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-15  122.813004  175.081985  137.149368  129.850006  98.585487   
2023-08-16  118.619865  174.213745  136.635483  128.830002  98.200302   
2023-08-17  118.121384  171.678055  135.256088  127.989998  98.576080   
2023-08-18  116.557503  172.161530  135.364243  125.059998  97.429955   
2023-08-21  116.176308  173.493500  135.382248  124.000000  97.580261   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-15  73.561508  291.048492  518.700012  172.738815  75.035553  ...   
2023-08-16  73.171638  291.275787  514.489990  168.785202  74.798882  ...   
2023-08-17  70.642250  289.382172  511.670013  167.184738  75.135704  ...   
2023-08-18  71.545601  286.825745  508.130005  167.432404  75.399681  ...   
2023-08-21  71.269844  290.205933  520.469971  168.832840  75.108398  ...   

Price        Volume                                                     \
Ticker           WY       WYNN      XEL       XOM        XYL       XYZ   
Date                                                                     
2023-08-15  2800100  3222100.0  3004000  14909600  1113300.0   9501100   
2023-08-16  2468700  2645500.0  3099300  13695100  1115700.0   8309200   
2023-08-17  2444400  1810500.0  4337500  16709500  1307400.0  10195200   
2023-08-18  2168200  1805900.0  3637000  16224000  1137300.0   9841100   
2023-08-21  2439100  2026000.0  3132200  11092400   773700.0   9422900   

Price                                              
Ticker            YUM      ZBH      ZBRA      ZTS  
Date                                               
2023-08-15  1247500.0   994800  501000.0  1001200  
2023-08-16  2082800.0   943800  374500.0  1205300  
2023-08-17  1582700.0  1007100  343200.0  2197100  
2023-08-18  1308800.0  1329600  699200.0  2027200  
2023-08-21  1537100.0  1248100  373900.0   860200  

[5 rows x 2515 columns]

In [9]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_1248/2048977385.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
496,2023-08-16,A,118.619865,-3.414246
497,2023-08-16,AAPL,174.213745,-0.495905
498,2023-08-16,ABBV,136.635483,-0.374690
499,2023-08-16,ABNB,128.830002,-0.785525
500,2023-08-16,ABT,98.200302,-0.390712
501,2023-08-16,ACGL,73.171638,-0.529991
502,2023-08-16,ACN,291.275787,0.078095
503,2023-08-16,ADBE,514.489990,-0.811649
504,2023-08-16,ADI,168.785202,-2.288781
505,2023-08-16,ADM,74.798882,-0.315412


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [12]:
fdf_appl = fdf[fdf['ticker'] == 'AAPL'].copy()
fdf_appl.head()

,date,ticker,close,return
497,2023-08-16,AAPL,174.213745,-0.495905
993,2023-08-17,AAPL,171.678055,-1.455505
1489,2023-08-18,AAPL,172.161530,0.281617
1985,2023-08-21,AAPL,173.493500,0.773675
2481,2023-08-22,AAPL,174.864944,0.790488


In [13]:
len(fdf_appl)

752

In [15]:
fdf_appl['Lag 1'] = fdf_appl['close'].shift(1)
fdf_appl['Lag 2'] = fdf_appl['close'].shift(2)
fdf_appl['Lag 3'] = fdf_appl['close'].shift(3)
fdf_appl['Lag 5'] = fdf_appl['close'].shift(5)
fdf_appl.head(25)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
497,2023-08-16,AAPL,174.213745,-0.495905,NaN,NaN,NaN,NaN
993,2023-08-17,AAPL,171.678055,-1.455505,174.213745,NaN,NaN,NaN
1489,2023-08-18,AAPL,172.161530,0.281617,171.678055,174.213745,NaN,NaN
1985,2023-08-21,AAPL,173.493500,0.773675,172.161530,171.678055,174.213745,NaN
2481,2023-08-22,AAPL,174.864944,0.790488,173.493500,172.161530,171.678055,NaN
2977,2023-08-23,AAPL,178.703018,2.194879,174.864944,173.493500,172.161530,174.213745
3473,2023-08-24,AAPL,174.026276,-2.617047,178.703018,174.864944,173.493500,171.678055
3969,2023-08-25,AAPL,176.226501,1.264307,174.026276,178.703018,174.864944,172.161530
4465,2023-08-28,AAPL,177.785492,0.884652,176.226501,174.026276,178.703018,173.493500
4961,2023-08-29,AAPL,181.662994,2.181000,177.785492,176.226501,174.026276,174.864944


In [17]:
fdf_appl['SMA 5'] = fdf_appl['close'].rolling(window=5).mean()
fdf_appl['SMA 10'] = fdf_appl['close'].rolling(window=10).mean()
fdf_appl['SMA 20'] = fdf_appl['close'].rolling(window=20).mean()
fdf_appl['SMA 30'] = fdf_appl['close'].rolling(window=30).mean()
fdf_appl['SMA 50'] = fdf_appl['close'].rolling(window=50).mean()

fdf_appl.head(100)
#fdf_appl = fdf[fdf['ticker'] == 'AAPL'].copy()
#fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
497,2023-08-16,AAPL,174.213745,-0.495905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
993,2023-08-17,AAPL,171.678055,-1.455505,174.213745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1489,2023-08-18,AAPL,172.161530,0.281617,171.678055,174.213745,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1985,2023-08-21,AAPL,173.493500,0.773675,172.161530,171.678055,174.213745,NaN,NaN,NaN,NaN,NaN,NaN
2481,2023-08-22,AAPL,174.864944,0.790488,173.493500,172.161530,171.678055,NaN,173.282355,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47678,2024-01-02,AAPL,183.404007,-3.578661,190.211014,191.248352,190.823547,191.268143,189.282336,191.059668,191.691473,190.490283,184.518847
48175,2024-01-03,AAPL,182.030777,-0.748746,183.404007,190.211014,191.248352,190.724762,187.543539,189.909692,191.435593,190.311135,184.748003
48672,2024-01-04,AAPL,179.718948,-1.270021,182.030777,183.404007,190.211014,190.823547,185.322620,188.424797,190.867026,189.996966,184.928553
49169,2024-01-05,AAPL,178.997711,-0.401314,179.718948,182.030777,183.404007,191.248352,182.872491,187.076236,190.316734,189.685430,185.085998


In [18]:
fdf_appl['volatility 5'] = fdf_appl['return'].rolling(5).std()
fdf_appl['volatility 10'] = fdf_appl['return'].rolling(10).std()
fdf_appl['volatility 20'] = fdf_appl['return'].rolling(20).std()
fdf_appl['volatility 30'] = fdf_appl['return'].rolling(30).std()
fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
497,2023-08-16,AAPL,174.213745,-0.495905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
993,2023-08-17,AAPL,171.678055,-1.455505,174.213745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1489,2023-08-18,AAPL,172.161530,0.281617,171.678055,174.213745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1985,2023-08-21,AAPL,173.493500,0.773675,172.161530,171.678055,174.213745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2481,2023-08-22,AAPL,174.864944,0.790488,173.493500,172.161530,171.678055,NaN,173.282355,NaN,NaN,NaN,NaN,0.957201,NaN,NaN,NaN


In [20]:
fdf_appl = fdf_appl.dropna()
#fdf_appl = fdf_appl.drop(columns = 'SMA 100')
fdf_appl.head()


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
24816,2023-10-25,AAPL,168.816727,-1.349172,171.125504,170.691406,170.572998,173.493500,170.865039,173.370158,173.081063,172.957731,174.848360,0.810499,0.692152,0.868370,1.021775
25313,2023-10-26,AAPL,164.662903,-2.460552,168.816727,171.125504,170.691406,173.118561,169.173907,172.006598,172.893597,172.666667,174.657343,1.139409,0.837129,1.029515,1.095777
25810,2023-10-27,AAPL,165.975159,0.796935,164.662903,168.816727,171.125504,170.572998,168.254340,170.957780,172.746092,172.443354,174.543285,1.335382,0.967925,1.045646,1.108552
26307,2023-10-30,AAPL,168.017548,1.230539,165.975159,164.662903,168.816727,170.691406,167.719568,170.126031,172.575402,172.190771,174.460405,1.550941,1.123099,1.027063,1.085458
26804,2023-10-31,AAPL,168.491135,0.281868,168.017548,165.975159,164.662903,171.125504,167.192694,169.496544,172.494990,171.917796,174.360358,1.553475,1.136789,1.017330,1.079065


In [21]:
len(fdf_appl)

703

In [22]:
fdf_appl['future return'] = fdf_appl.groupby('ticker')['return'].shift(-1)
fdf_appl = fdf_appl.dropna()
print(len(fdf_appl))
fdf_appl.head(len(fdf_appl))


702


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
24816,2023-10-25,AAPL,168.816727,-1.349172,171.125504,170.691406,170.572998,173.493500,170.865039,173.370158,173.081063,172.957731,174.848360,0.810499,0.692152,0.868370,1.021775,-2.460552
25313,2023-10-26,AAPL,164.662903,-2.460552,168.816727,171.125504,170.691406,173.118561,169.173907,172.006598,172.893597,172.666667,174.657343,1.139409,0.837129,1.029515,1.095777,0.796935
25810,2023-10-27,AAPL,165.975159,0.796935,164.662903,168.816727,171.125504,170.572998,168.254340,170.957780,172.746092,172.443354,174.543285,1.335382,0.967925,1.045646,1.108552,1.230539
26307,2023-10-30,AAPL,168.017548,1.230539,165.975159,164.662903,168.816727,170.691406,167.719568,170.126031,172.575402,172.190771,174.460405,1.550941,1.123099,1.027063,1.085458,0.281868
26804,2023-10-31,AAPL,168.491135,0.281868,168.017548,165.975159,164.662903,171.125504,167.192694,169.496544,172.494990,171.917796,174.360358,1.553475,1.136789,1.017330,1.079065,1.873878
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372872,2026-08-07,AAPL,313.059998,0.294479,312.140808,310.732025,309.113403,308.643829,309.640961,320.429651,323.152800,316.219614,309.518459,1.337704,2.643575,2.357699,2.252916,-1.533249
373375,2026-08-10,AAPL,308.260010,-1.533249,313.059998,312.140808,310.732025,303.158569,310.661249,317.593683,322.713972,317.043765,309.438845,1.244582,2.582995,2.376576,2.222654,-1.086747
373877,2026-08-11,AAPL,304.910004,-1.086747,308.260010,313.059998,312.140808,309.113403,309.820569,314.105991,322.230038,317.824191,309.301223,0.966260,2.505644,2.382203,2.229492,-0.872390
374380,2026-08-12,AAPL,302.250000,-0.872390,304.910004,308.260010,313.059998,310.732025,308.124164,310.541132,320.981648,318.262169,309.225302,0.877313,2.500730,2.176626,2.191281,0.995868


In [23]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50']
x = fdf_appl[features]
y = fdf_appl['future return']



In [24]:
split_index = int(0.8 * len(fdf_appl))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [25]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
tree = DecisionTreeRegressor(max_depth = 5, random_state=72)

tree.fit(x_train, y_train)
predictions = tree.predict(x_test)

score = mean_squared_error(y_test, predictions)
score


6.665599134822851

In [26]:
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor(n_estimators = 500, random_state=72)
forest.fit(x_train, y_train)

RandomForestRegressor(n_estimators=500, random_state=72)

In [27]:
predictions = forest.predict(x_test)
score = mean_squared_error(y_test, predictions)
score

3.5608099937210187